In [ ]:
!pip install SpeechRecognition sounddevice soundfile numpy pydub
!pip install gTTS
!pip install jiwer pymorphy2 num2words

In [ ]:
from gtts import gTTS
import os
import soundfile as sf
from IPython.display import Audio, display
import numpy as np
from pathlib import Path
import speech_recognition as spr
import io
from pydub import AudioSegment
from jiwer import wer, cer
import re
from num2words import num2words

In [ ]:
directory = Path("clear")
if not directory.exists():
    directory.mkdir(parents=True)
texts = ["Привет, как дела?  Сегодня отличная погода Проверка микрофона один два три четыре Запись тестового аудио"]
for i, t in enumerate(texts):
    gTTS(text=t, lang='ru').save(f"clear/sample_{i}.mp3")

In [ ]:
files_paths = ['clear/sample_0.mp3', 'clear/sample_1.mp3', 'clear/sample_2.mp3', 'clear/sample_3.mp3', 'clear/sample_4.mp3', ]

for file_path in files_paths:
    audio = Audio(file_path)
    display(audio)

In [ ]:


def add_noise(audio, snr_db=10, noise_type="white"):
    """
    audio: numpy array (float32, нормализован [-1, 1])
    snr_db: отношение сигнал/шум в дБ
    noise_type: 'white' или 'pink'
    """
    signal_power = np.mean(audio**2)

    if noise_type == "pink":
        # Розовый шум (спад 3 дБ/октаву, более естественный)
        white = np.random.randn(len(audio))
        pink = np.cumsum(white) / np.sqrt(np.arange(1, len(audio)+1))
        noise = pink / np.std(pink)  # нормализуем
    else:
        noise = np.random.randn(len(audio))

    noise_power = np.mean(noise**2)
    # Масштабируем шум под нужный SNR
    noise_scaled = noise * np.sqrt(signal_power / (noise_power * 10**(snr_db/10)))

    noisy = audio + noise_scaled

    # Защита от клиппинга
    max_val = np.max(np.abs(noisy))
    if max_val > 1.0:
        noisy /= max_val

    return noisy.astype(np.float32)

In [ ]:
noise_files_paths=[]
out_dir = Path("noisy_audio")
if not out_dir.exists():
    out_dir.mkdir(parents=True)
for i, file in enumerate(files_paths[:5]):
    audio, sr = sf.read(file)
    if audio.ndim > 1:
        audio = audio[:, 0]

    snr = -10
    noisy = add_noise(audio, snr_db=snr)

    out_name = out_dir / f"{Path(file).stem}_snr{snr}.mp3"
    sf.write(out_name, noisy, sr)
    noise_files_paths.append(out_name)
    print(f"🔊 {file} → {out_name} (SNR={snr} дБ)")

🔊 clear/sample_0.mp3 → noisy_audio/sample_0_snr-10.mp3 (SNR=-10 дБ)
🔊 clear/sample_1.mp3 → noisy_audio/sample_1_snr-10.mp3 (SNR=-10 дБ)
🔊 clear/sample_2.mp3 → noisy_audio/sample_2_snr-10.mp3 (SNR=-10 дБ)
🔊 clear/sample_3.mp3 → noisy_audio/sample_3_snr-10.mp3 (SNR=-10 дБ)
🔊 clear/sample_4.mp3 → noisy_audio/sample_4_snr-10.mp3 (SNR=-10 дБ)


In [ ]:
for file_path in noise_files_paths:
    audio = Audio(file_path)
    display(audio)

In [ ]:
def post_process_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)

    # Разделение на предложения и перевод первой буквы в верхний регистр
    sentences = re.split(r'([.!?])\s*', text)
    processed_sentences = []

    for i in range(0, len(sentences)-1, 2):
        sentence = sentences[i].strip()
        if sentence:
            # Перевод первой буквы предложения в верхний регистр
            sentence = sentence[0].upper() + sentence[1:] if sentence else sentence
            # Добавление знака препинания и перенос строки
            processed_sentences.append(sentence + sentences[i+1] + '\n')

    # Обработка последнего предложения, если оно есть
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        last_sentence = sentences[-1].strip()
        last_sentence = last_sentence[0].upper() + last_sentence[1:]
        processed_sentences.append(last_sentence)

    text = ''.join(processed_sentences)

    # Исправление частых ошибок
    common_errors = {
        'щто': 'что',
        'чьто': 'что',
        'шта': 'что',
        'щта': 'что',
        'ево': 'его',
        'каво': 'кого',
        'тобишь': 'то есть',
        'патаму': 'потому',
        'щас': 'сейчас'
    }
    for error, correction in common_errors.items():
        text = re.sub(r'\b' + error + r'\b', correction, text)

    # Обработка чисел
    def process_number(match):
        try:
            return num2words(int(match.group(0)), lang='ru')
        except:
            return match.group(0)
    text = re.sub(r'\b\d+\b', process_number, text)

    return text.strip()

In [ ]:
def download_and_convert_audio(audio_file):

    audio = AudioSegment.from_mp3(audio_file)
    wav_io = io.BytesIO()
    audio.export(wav_io, format='wav')
    wav_io.seek(0)
    return wav_io

def recognize_from_file(path):
    # Инициализация распознавателя
    recognizer = spr.Recognizer()

    # Загрузка и конвертация файла
    wav_file = download_and_convert_audio(path)

    # Распознавание из файла
    with spr.AudioFile(wav_file) as source:
        print("Загрузка аудио...")
        audio = recognizer.record(source)

    try:
        # Распознавание с помощью Google Speech Recognition
        text = post_process_text(recognizer.recognize_google(audio, language="ru-RU"))
        print(f"Распознанный текст: {text}")
        return text
    except spr.UnknownValueError:
        print("Не удалось распознать речь")
        return('error')
    except spr.RequestError as e:
        print(f"Ошибка сервиса распознавания: {e}")
        return('error')

In [ ]:
clear_text = []
noise_text = []

print('файлы без шума')
for path in files_paths:
  clear_text.append(recognize_from_file(path))

print('файлы с шумом')
for path in noise_files_paths:
  noise_text.append(recognize_from_file(path))

файлы без шума
Загрузка аудио...
Распознанный текст: Привет как дела
Загрузка аудио...
Распознанный текст: Сегодня отличная погода
Загрузка аудио...
Распознанный текст: Проверка микрофона
Загрузка аудио...
Распознанный текст: один два три четыре
Загрузка аудио...
Распознанный текст: Запись тестового аудио
файлы с шумом
Загрузка аудио...
Распознанный текст: Привет как
Загрузка аудио...
Распознанный текст: Сегодня отличная погода
Загрузка аудио...
Распознанный текст: Проверка
Загрузка аудио...
Распознанный текст: один два три четыре
Загрузка аудио...
Не удалось распознать речь


In [ ]:
for i,text in enumerate(texts):
  reference = text
  hypothesis = clear_text[i]
  print(f'Исходный текст: {reference}')

  print(f'Распознанный текст: {hypothesis}')
  wer_value = wer(reference, hypothesis)
  print(f"WER: {wer_value}")

  # Пример вычисления CER
  cer_value = cer(reference, hypothesis)
  print(f"CER: {cer_value}")
  print(' ')

Исходный текст: Привет, как дела?
Распознанный текст: Привет как дела
WER: 0.6666666666666666
CER: 0.11764705882352941
 
Исходный текст: Сегодня отличная погода
Распознанный текст: Сегодня отличная погода
WER: 0.0
CER: 0.0
 
Исходный текст: Проверка микрофона
Распознанный текст: Проверка микрофона
WER: 0.0
CER: 0.0
 
Исходный текст: один два три четыре
Распознанный текст: один два три четыре
WER: 0.0
CER: 0.0
 
Исходный текст: Запись тестового аудио
Распознанный текст: Запись тестового аудио
WER: 0.0
CER: 0.0
 


In [ ]:
print('Распосзнование с шумом')
for i,text in enumerate(texts):
  reference = text
  hypothesis = noise_text[i]
  print(f'Исходный текст: {reference}')

  print(f'Распознанный текст: {hypothesis}')
  wer_value = wer(reference, hypothesis)
  print(f"WER: {wer_value}")

  # Пример вычисления CER
  cer_value = cer(reference, hypothesis)
  print(f"CER: {cer_value}")
  print(' ')

Распосзнование с шумом
Исходный текст: Привет, как дела?
Распознанный текст: Привет как
WER: 0.6666666666666666
CER: 0.4117647058823529
 
Исходный текст: Сегодня отличная погода
Распознанный текст: Сегодня отличная погода
WER: 0.0
CER: 0.0
 
Исходный текст: Проверка микрофона
Распознанный текст: Проверка
WER: 0.5
CER: 0.5555555555555556
 
Исходный текст: один два три четыре
Распознанный текст: один два три четыре
WER: 0.0
CER: 0.0
 
Исходный текст: Запись тестового аудио
Распознанный текст: error
WER: 1.0
CER: 1.0
 


Для пробных записей я решил пойти простым путём генерации голоса самым доступным методом который смог найти

Добавлять шум тоже решил довольно простым способом а именно простым белым шумом

По итогу добавления шума запись стала куда сложнее различима

Хоть и белый шум не отражает условие реалистичности шума который может быть в более реальных условиях

По итогу библиотка SpeechRecognition отлично справилась с распознованием чистых записей точность метрик снижена только из-за отсутвия знаков припинания в первом экземпляре

В остальных примерах результат отличный

При добавлении шума библиотека справилась хуже

Для меня удивительно что цифры были распознаны так же хорошо как и при чистой записи

Хотя на мой слух их слышно довольно плохо


